유저 세션 Apply 유입 경로 분석

이 노트북은 다음 로직을 수행합니다:
1. 세션 정의: 유저별로 30분 이상의 유휴 시간이 발생하면 새로운 세션으로 구분합니다.
2. 타겟 필터링: `POST` 메서드이면서 `api/jobs/id/apply/` 경로인 '사용자의 지원 액션'을 추출합니다.
3. 경로 추적: 각 지원 액션 바로 직전에 발생한 이벤트(Previous Event)를 찾아 유입 경로를 파악합니다.

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import glob
import pandas as pd
import datetime
import numpy as np
import polars as pl
from pathlib import Path

In [3]:
# Parquet 파일 경로 지정 
DATA_DIR = Path(r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\DBsources\topic1")

df = pl.read_parquet(DATA_DIR / 'cleaned_log_data.parquet')

# 0. 원본 데이터 행 수 출력
print(f"📊 [검증 0] 원본 데이터 행 수: {len(df):,} 개")

📊 [검증 0] 원본 데이터 행 수: 17,240,278 개


In [4]:
import polars as pl

# ---------------------------------------------------------
# T(Transform): timestamp 컬럼을 명확한 포맷으로 Datetime 타입 변환
# ---------------------------------------------------------

# ---------------------------------------------------------
# 기존 파이프라인 (정렬 및 세션화)
# ---------------------------------------------------------
df_sessionized = (
    df.sort(["user_uuid", "timestamp"])
    .with_columns(
        pl.col("timestamp").diff().over("user_uuid").alias("time_diff")
    )
    .with_columns(
        (pl.col("time_diff").dt.total_milliseconds() / (1000 * 60)).round(2).alias("time_diff_minutes")
    )
    .with_columns(
        (pl.col("time_diff") > pl.duration(minutes=30)).fill_null(False).alias("is_new_session")
    )
    .with_columns(
        pl.col("is_new_session").cum_sum().over("user_uuid").alias("session_id")
    )
)

print(f"\n📊 [검증 1] 세션화 완료 데이터 행 수: {len(df_sessionized):,} 개")

# 🔍 잘 변환되었는지 눈으로 직접 확인해 보기
print("\n🔍 변환된 timestamp와 time_diff 샘플 (정상 작동 확인용):")
print(df_sessionized.select(["user_uuid", "timestamp", "time_diff", "session_id"]).head(10))


📊 [검증 1] 세션화 완료 데이터 행 수: 17,240,278 개

🔍 변환된 timestamp와 time_diff 샘플 (정상 작동 확인용):
shape: (10, 4)
┌──────────────────────────────┬────────────────────────────┬─────────────────────────┬────────────┐
│ user_uuid                    ┆ timestamp                  ┆ time_diff               ┆ session_id │
│ ---                          ┆ ---                        ┆ ---                     ┆ ---        │
│ str                          ┆ datetime[μs]               ┆ duration[μs]            ┆ u32        │
╞══════════════════════════════╪════════════════════════════╪═════════════════════════╪════════════╡
│ 0002535c-eacb-456b-a620-92c9 ┆ 2022-01-15 07:44:06.150657 ┆ null                    ┆ 0          │
│ 17…                          ┆                            ┆                         ┆            │
│ 0002535c-eacb-456b-a620-92c9 ┆ 2022-01-15 07:44:08.578129 ┆ 2s 427472µs             ┆ 0          │
│ 17…                          ┆                            ┆                         ┆       

In [5]:
# 🔍 잘 변환되었는지 눈으로 직접 확인해 보기
print("\n🔍 변환된 timestamp와 time_diff 샘플 (정상 작동 확인용):")
print(df_sessionized.select(["user_uuid", "timestamp", "time_diff", "time_diff_minutes", "session_id"]).head(10))


🔍 변환된 timestamp와 time_diff 샘플 (정상 작동 확인용):
shape: (10, 5)
┌───────────────────────┬─────────────────┬───────────────────────┬───────────────────┬────────────┐
│ user_uuid             ┆ timestamp       ┆ time_diff             ┆ time_diff_minutes ┆ session_id │
│ ---                   ┆ ---             ┆ ---                   ┆ ---               ┆ ---        │
│ str                   ┆ datetime[μs]    ┆ duration[μs]          ┆ f64               ┆ u32        │
╞═══════════════════════╪═════════════════╪═══════════════════════╪═══════════════════╪════════════╡
│ 0002535c-eacb-456b-a6 ┆ 2022-01-15      ┆ null                  ┆ null              ┆ 0          │
│ 20-92c917…            ┆ 07:44:06.150657 ┆                       ┆                   ┆            │
│ 0002535c-eacb-456b-a6 ┆ 2022-01-15      ┆ 2s 427472µs           ┆ 0.04              ┆ 0          │
│ 20-92c917…            ┆ 07:44:08.578129 ┆                       ┆                   ┆            │
│ 0002535c-eacb-456b-a6 ┆ 2022-0

In [6]:
# 1. 특정 이벤트(Apply) 플래그 생성 (이 부분은 동일)
df_apply_flagged = df_sessionized.with_columns(
    (
        (pl.col("method") == "POST") & 
        (pl.col("URL").str.contains(r"api/jobs/.+/apply"))
    ).alias("is_apply_event")
)

# ---------------------------------------------------------
# 🚨 핵심 변경: over()를 쓰지 않고 타겟 세션의 '키(Key)'만 먼저 추출
# ---------------------------------------------------------
valid_sessions = (
    df_apply_flagged
    .filter(pl.col("is_apply_event") == True)
    .select(["user_uuid", "session_id"])
    .unique() # 중복 제거하여 딱 조건에 맞는 (유저, 세션) 목록만 남김
)

# ---------------------------------------------------------
# 원본과 Inner Join하여 타겟 세션의 전체 로그만 가볍게 가져옴
# ---------------------------------------------------------
df_target_sessions = df_apply_flagged.join(
    valid_sessions, 
    on=["user_uuid", "session_id"], 
    how="inner"
)

print(f"📊 [파이프라인 3] Apply 이벤트가 포함된 타겟 세션의 총 로그 수: {len(df_target_sessions):,} 개")

# 2. user 한 명의 세션 예시 출력 (검증)
if len(df_target_sessions) > 0:
    # 첫 번째 유저와 그 유저의 세션 번호를 가져옴
    sample_user = df_target_sessions["user_uuid"][0]
    sample_session = df_target_sessions["session_id"][0]
    
    sample_df = df_target_sessions.filter(
        (pl.col("user_uuid") == sample_user) & 
        (pl.col("session_id") == sample_session)
    ).select(["timestamp", "method", "URL", "time_diff", "is_apply_event"])
    
    print(f"\n🔍 [검증 2] User: {sample_user} | Session: {sample_session} 의 행동 패턴:")
    print(sample_df)
else:
    print("\n조건에 맞는 세션이 없습니다. (메소드명이나 URL 패턴을 확인해 주세요)")

📊 [파이프라인 3] Apply 이벤트가 포함된 타겟 세션의 총 로그 수: 4,913,185 개

🔍 [검증 2] User: 0002535c-eacb-456b-a620-92c917332ba3 | Session: 7 의 행동 패턴:
shape: (11, 5)
┌─────────────────┬────────┬───────────────────────────┬──────────────────────────┬────────────────┐
│ timestamp       ┆ method ┆ URL                       ┆ time_diff                ┆ is_apply_event │
│ ---             ┆ ---    ┆ ---                       ┆ ---                      ┆ ---            │
│ datetime[μs]    ┆ str    ┆ str                       ┆ duration[μs]             ┆ bool           │
╞═════════════════╪════════╪═══════════════════════════╪══════════════════════════╪════════════════╡
│ 2022-06-11      ┆ GET    ┆ continue?next=/jobs/12894 ┆ 3d 16h 1m 16s 366680µs   ┆ false          │
│ 23:05:45.340542 ┆        ┆ 2/app…                    ┆                          ┆                │
│ 2022-06-11      ┆ GET    ┆ jobs/id/apply/step1       ┆ 272079µs                 ┆ false          │
│ 23:05:45.612621 ┆        ┆                    

In [7]:
# 1. 세션별로 첫 번째 Apply(지원)가 일어난 시점을 찾습니다.
# target_sessions는 이미 Apply가 포함된 세션만 모여있는 상태입니다.
first_apply_moments = (
    df_target_sessions
    .filter(pl.col("is_apply_event") == True)
    .group_by(["user_uuid", "session_id"])
    .agg(pl.col("timestamp").min().alias("first_apply_time"))
)

# 2. 전체 로그와 조인하여 '지원 시점 이전'의 로그만 남깁니다.
df_before_apply = (
    df_target_sessions
    .join(first_apply_moments, on=["user_uuid", "session_id"], how="inner")
    .filter(pl.col("timestamp") < pl.col("first_apply_time"))
)

# 3. 지원 직전 어떤 페이지(URL)를 가장 많이 방문했는지 TOP 10 추출
top_paths = (
    df_before_apply
    .group_by("URL")
    .len(name="visit_count")
    .sort("visit_count", descending=True)
    .head(10)
)

print("📊 [분석 완료] 지원(Apply) 직전 가장 많이 방문한 이전 경로 Top 10:")
print(top_paths)

📊 [분석 완료] 지원(Apply) 직전 가장 많이 방문한 이전 경로 Top 10:
shape: (10, 2)
┌─────────────────────────────────┬─────────────┐
│ URL                             ┆ visit_count │
│ ---                             ┆ ---         │
│ str                             ┆ u32         │
╞═════════════════════════════════╪═════════════╡
│ api/recommend_specialty         ┆ 207026      │
│ jobs/id/id_title                ┆ 159519      │
│ api/jobs/id/other_jobs?offset=… ┆ 129613      │
│ api/users/id/template           ┆ 82052       │
│ jobs/id/apply/step1             ┆ 65532       │
│ companies/company_id/jobs       ┆ 49216       │
│ @user_id                        ┆ 48524       │
│ api/users/id/experience/form?t… ┆ 48250       │
│ null                            ┆ 44509       │
│ jobs                            ┆ 38490       │
└─────────────────────────────────┴─────────────┘


In [8]:
output_path = r"C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\notebooks\test_results\refined_apply_sessions.parquet"

# 분석에 사용된 모든 정보를 포함하여 저장
df_target_sessions.write_parquet(output_path)

print(f"✅ [검증 3] 분석용 세션 데이터 저장 완료!")
print(f"파일 위치: {output_path}")
print(f"총 행 수: {len(df_target_sessions):,} 개")

✅ [검증 3] 분석용 세션 데이터 저장 완료!
파일 위치: C:\Users\user\Desktop\playground\codeit_DA_13th\projects\intermediate1\midproject1\notebooks\test_results\refined_apply_sessions.parquet
총 행 수: 4,913,185 개
